# 📊 Gráficos — Digitalización y Posicionamiento de Restaurantes Mediterráneos en Bogotá
**Haciendo Economía 2026-1 · Universidad del Rosario**

Este notebook genera todos los gráficos de la Tabla 1 del checklist (excepto mapas).
Cada gráfico se guarda también como `.html` interactivo en la misma carpeta.

> **Requisitos:** `pip install plotly pandas openpyxl nbformat`

## ⚙️ Setup — paleta, librerías y datos

In [7]:
#%pip install plotly pandas openpyxl nbformat
import os

# ── Ruta de salida ────────────────────────────────────────────────────
BASE=r"C:\Users\anton\OneDrive\Documentos\GitHub\DoingEconomics_Proyect\Data\Clean\tabla_indices.xlsx"
OUTPUT_PATH = r"C:\Users\anton\OneDrive\Documents\GitHub\DoingEconomics_Proyect\Outputs\Graphs"
os.makedirs(OUTPUT_PATH, exist_ok=True)
print(f"✓ Carpeta de salida: {OUTPUT_PATH}")


import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings("ignore")

# ── Paleta ────────────────────────────────────────────────────────────────────
MOSTAZA   = "#F2C500"
CORAL     = "#EA5A45"
AZUL_C    = "#86A8CC"
AZUL_P    = "#4D6B99"
VERDE_S   = "#94C56E"
VERDE_A   = "#2C817C"
GRIS_F    = "#F7F6F3"
GRIS_T    = "#4A4A4A"
GRIS_M    = "#9E9E9E"

# Color por nivel (bajo/medio/alto)
NIVEL_COLOR = {"Bajo (0-3)": CORAL, "Bajo (0-9)": CORAL, "Bajo (0-5)": CORAL,
               "Medio (4-6)": MOSTAZA, "Medio (5-9)": MOSTAZA, "Medio (6-11)": MOSTAZA,
               "Alto (7-10)": VERDE_A, "Alto (10-14)": VERDE_A, "Alto (12-18)": VERDE_A}

# Tipografía base
FONT = dict(family="Inter, Arial, sans-serif", size=13, color=GRIS_T)

def base_layout(title="", height=480, showlegend=True):
    return dict(
        title=dict(text=title, font=dict(size=16, color=GRIS_T, family="Inter, Arial, sans-serif"),
                   x=0.03, xanchor="left", y=0.97),
        font=FONT,
        paper_bgcolor="white",
        plot_bgcolor=GRIS_F,
        height=height,
        showlegend=showlegend,
        margin=dict(l=48, r=32, t=72, b=48),
        legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1,
                    font=dict(size=12)),
    )

# ── Cargar datos ──────────────────────────────────────────────────────────────
df = pd.read_excel(BASE)
print(f"✓ Base cargada: {df.shape[0]} restaurantes · {df.shape[1]} variables")
df.head(2)


✓ Carpeta de salida: C:\Users\anton\OneDrive\Documents\GitHub\DoingEconomics_Proyect\Outputs\Graphs
✓ Base cargada: 35 restaurantes · 91 variables


,"Restaurante con encuesta aplicada (1=Si, 0=No)",hot_zone,Identificador del restaurante,Nombre del restaurante,Barrio,p0a - Tipo de cocina,p3 - Pertenece a cadena,p3b - Competitividad percibida (1-5),p4 - Realiza inversion en marketing digital,p5 - Plataformas donde invierte en publicidad paga,...,"p9 - De cada 10 clientes, cuantos llegan de manera presencial","p10 - De cada 10 clientes, cuantos llegan por delivery (plataformas)","p11 - De cada 10 clientes, cuantos llegan por domicilio propio",p12 - Canal percibido como mas rentable,p12b - Acceso y consulta de datos de comportamiento en delivery,"p12c - Uso de datos de delivery para decisiones de menu, precios o promociones",p7b_contenido,p7b_programac,p7b_algoritm,ia_integrad
0,1,1,1,Vapiano Colombia Restaurante Italiano,Zona T,Italiana,Si,4,Si,"Facebook / Instagram Ads, Google Ads, TikTok Ads",...,4,4,2,Delivery por plataformas,"Sí, los consulto regularmente","Sí, con frecuencia",1,0,0,1
1,1,1,2,Storia D'Amore zona T,Zona T,Italiana,Si,3,Si,"Facebook / Instagram Ads, Google Ads, TikTok Ads",...,4,4,0,Presencial,"Sí, los consulto regularmente","Sí, ocasionalmente",1,1,1,1


## G2 · Distribución de muestra por tipo de cocina
*Escena 4 — dona o barras horizontales*

In [8]:

# ── G2 · Distribución de muestra por tipo de cocina (E4) ─────────────────────
cocina = df["p0a - Tipo de cocina"].value_counts().reset_index()
cocina.columns = ["Tipo", "n"]
cocina["pct"] = (cocina["n"] / cocina["n"].sum() * 100).round(1)

colors = [AZUL_P, VERDE_A, MOSTAZA, CORAL]

fig2 = go.Figure(go.Pie(
    labels=cocina["Tipo"],
    values=cocina["n"],
    hole=0.58,
    marker=dict(colors=colors[:len(cocina)], line=dict(color="white", width=2.5)),
    textinfo="label+percent",
    textfont=dict(size=13, family="Inter, Arial, sans-serif"),
    insidetextorientation="radial",
    sort=False,
))

fig2.add_annotation(text=f"<b>n = {cocina['n'].sum()}</b><br>restaurantes",
                    x=0.5, y=0.5, showarrow=False,
                    font=dict(size=15, color=GRIS_T, family="Inter, Arial, sans-serif"),
                    align="center")

fig2.update_layout(**base_layout("Muestra por tipo de cocina", height=420, showlegend=False))
fig2.update_traces(pull=[0.03]*len(cocina))
fig2.show()
fig2.write_html(os.path.join(OUTPUT_PATH, "G2_cocina.html"))
print("✓ G2 guardado")


✓ G2 guardado


## G3 · % inversión en marketing digital
*Escena 5 — indicador único*

In [9]:

# ── G3 · % que invierte en marketing digital (E5) ────────────────────────────
si = (df["p4 - Realiza inversion en marketing digital"] == "Si").sum()
pct = round(si / len(df) * 100, 1)

fig3 = go.Figure()

# Barra de fondo
fig3.add_trace(go.Bar(x=[""], y=[100], marker_color="#E8E8E8", width=0.35,
                      showlegend=False, hoverinfo="skip"))
# Barra real
fig3.add_trace(go.Bar(x=[""], y=[pct], marker_color=VERDE_A, width=0.35,
                      showlegend=False, name="Con inversión"))

fig3.add_annotation(text=f"<b>{pct}%</b>", x=0, y=pct/2,
                    showarrow=False, font=dict(size=42, color="white",
                    family="Inter, Arial, sans-serif"), align="center")
fig3.add_annotation(text="invierte activamente<br>en marketing digital",
                    x=0, y=pct/2 - 18, showarrow=False,
                    font=dict(size=14, color="white", family="Inter, Arial, sans-serif"),
                    align="center")

fig3.update_layout(**base_layout("Inversión en marketing digital", height=400),
                   xaxis=dict(visible=False), yaxis=dict(visible=False, range=[0, 110]),
                   barmode="overlay")
fig3.show()
fig3.write_html(os.path.join(OUTPUT_PATH, "G3_inversion_mkt.html"))
print("✓ G3 guardado")


✓ G3 guardado


## G4 · Plataformas de publicidad paga
*Escena 5 — barras horizontales ordenadas*

In [10]:

# ── G4 · Plataformas de publicidad paga (E5) ─────────────────────────────────
# ── G4 · Plataformas de publicidad paga (E5) ─────────────────────────────────
plataformas = {
    "FB / IG Ads":  (df["p5 - FB/IG Ads (dummy)"] == "Si").sum(),
    "Google Ads":   (df["p5 - Google Ads (dummy)"] == "Si").sum(),
    "TikTok Ads":   (df["p5 - TikTok Ads (dummy)"] == "Si").sum(),
}
plat_df = pd.DataFrame(list(plataformas.items()), columns=["Plataforma", "n"])
plat_df["pct"] = (plat_df["n"] / len(df) * 100).round(1)
plat_df = plat_df.sort_values("pct", ascending=True)

colors_p = [AZUL_C, AZUL_P, VERDE_A]

fig4 = go.Figure(go.Bar(
    x=plat_df["pct"], y=plat_df["Plataforma"],
    orientation="h",
    marker=dict(color=colors_p, line=dict(color="white", width=1)),
    text=[f"{v}%" for v in plat_df["pct"]],
    textposition="outside",
    textfont=dict(size=13, color=GRIS_T),
))

fig4.update_layout(**base_layout("Plataformas de publicidad paga", height=340, showlegend=False),
                   xaxis=dict(title="% de restaurantes", range=[0, 85],
                              showgrid=True, gridcolor="#E8E8E8", zeroline=False),
                   yaxis=dict(showgrid=False))
fig4.show()
fig4.write_html(os.path.join(OUTPUT_PATH, "G4_plataformas.html"))
print("✓ G4 guardado")


✓ G4 guardado


## G5 · Distribución IRS — Índice de Redes Sociales
*Escena 5 — dona bajo/medio/alto*

In [11]:

# ── G5 · Distribución IRS (E5) ───────────────────────────────────────────────
irs_counts = df["Nivel IRS"].value_counts()
order = ["Alto (10-14)", "Medio (5-9)", "Bajo (0-4)"]
# Usar el nivel que exista en los datos
order_real = [o for o in order if o in irs_counts.index]
irs_df = pd.DataFrame({"Nivel": order_real, "n": [irs_counts.get(o, 0) for o in order_real]})
irs_df["pct"] = (irs_df["n"] / irs_df["n"].sum() * 100).round(1)

colors_irs = [VERDE_A, MOSTAZA, CORAL]

fig5 = go.Figure(go.Pie(
    labels=irs_df["Nivel"], values=irs_df["n"],
    hole=0.60,
    marker=dict(colors=colors_irs[:len(irs_df)], line=dict(color="white", width=2.5)),
    textinfo="label+percent",
    textfont=dict(size=12, family="Inter, Arial, sans-serif"),
    sort=False,
))
fig5.add_annotation(text="<b>IRS</b><br>Redes sociales",
                    x=0.5, y=0.5, showarrow=False,
                    font=dict(size=14, color=GRIS_T, family="Inter, Arial, sans-serif"),
                    align="center")
fig5.update_layout(**base_layout("Índice de Redes Sociales — distribución", height=420, showlegend=True))
fig5.show()
fig5.write_html(os.path.join(OUTPUT_PATH, "G5_dona_IRS.html"))
print("✓ G5 guardado")


✓ G5 guardado


## G6 · IRS por zona de concentración
*Escena 5 — barras apiladas hot_zone vs. dispersa*

In [12]:

# ── G6 · IRS por zona de concentración (E5) ──────────────────────────────────
irs_zona = df.groupby(["hot_zone", "Nivel IRS"]).size().reset_index(name="n")
totals = df.groupby("hot_zone").size().reset_index(name="total")
irs_zona = irs_zona.merge(totals, on="hot_zone")
irs_zona["pct"] = (irs_zona["n"] / irs_zona["total"] * 100).round(1)
irs_zona["zona_label"] = irs_zona["hot_zone"].map({1: "Zona concentrada", 0: "Zona dispersa"})

order = ["Alto (10-14)", "Medio (5-9)", "Bajo (0-4)"]
order_real = [o for o in order if o in irs_zona["Nivel IRS"].unique()]
color_map = {"Alto (10-14)": VERDE_A, "Medio (5-9)": MOSTAZA, "Bajo (0-4)": CORAL}

fig6 = go.Figure()
for nivel in order_real:
    sub = irs_zona[irs_zona["Nivel IRS"] == nivel]
    fig6.add_trace(go.Bar(
        name=nivel, x=sub["zona_label"], y=sub["pct"],
        marker_color=color_map.get(nivel, AZUL_C),
        text=[f"{v}%" for v in sub["pct"]],
        textposition="inside", textfont=dict(color="white", size=13),
    ))

fig6.update_layout(**base_layout("IRS · Zona concentrada vs. dispersa", height=420),
                   barmode="stack",
                   xaxis=dict(showgrid=False),
                   yaxis=dict(title="% de restaurantes", showgrid=True,
                              gridcolor="#E8E8E8", range=[0, 105]))
fig6.show()
fig6.write_html(os.path.join(OUTPUT_PATH, "G6_IRS_zona.html"))
print("✓ G6 guardado")


✓ G6 guardado


## G · Distribución IDL — Índice de Delivery
*Escena 6 — dona bajo/medio/alto*

In [13]:

# ── G · Distribución IDL (E6) ────────────────────────────────────────────────
idl_counts = df["Nivel IDL"].value_counts()
order_idl = ["Alto (7-10)", "Medio (4-6)", "Bajo (0-3)"]
order_idl_real = [o for o in order_idl if o in idl_counts.index]
idl_df = pd.DataFrame({"Nivel": order_idl_real,
                        "n": [idl_counts.get(o, 0) for o in order_idl_real]})
idl_df["pct"] = (idl_df["n"] / idl_df["n"].sum() * 100).round(1)

fig_idl = go.Figure(go.Pie(
    labels=idl_df["Nivel"], values=idl_df["n"],
    hole=0.60,
    marker=dict(colors=[VERDE_A, MOSTAZA, CORAL][:len(idl_df)],
                line=dict(color="white", width=2.5)),
    textinfo="label+percent",
    textfont=dict(size=12, family="Inter, Arial, sans-serif"),
    sort=False,
))
fig_idl.add_annotation(text="<b>IDL</b><br>Delivery",
                        x=0.5, y=0.5, showarrow=False,
                        font=dict(size=14, color=GRIS_T, family="Inter, Arial, sans-serif"),
                        align="center")
fig_idl.update_layout(**base_layout("Índice de Delivery — distribución", height=420, showlegend=True))
fig_idl.show()
fig_idl.write_html(os.path.join(OUTPUT_PATH, "G_dona_IDL.html"))
print("✓ G_IDL guardado")


✓ G_IDL guardado


## G7 · Mix de ventas por canal
*Escena 6 — barras apiladas presencial / delivery / domicilio*

In [14]:

# ── G7 · Mix de ventas por canal (E6) ────────────────────────────────────────
canales = {
    "Presencial":      df["p9 - Clientes presenciales normalizados (de cada 10)"].mean(),
    "Delivery":        df["p10 - Clientes delivery normalizados (de cada 10)"].mean(),
    "Domicilio propio":df["p11 - Clientes domicilio propio normalizados (de cada 10)"].mean(),
}
canal_df = pd.DataFrame(list(canales.items()), columns=["Canal", "Promedio"])
canal_df["pct"] = (canal_df["Promedio"] / canal_df["Promedio"].sum() * 100).round(1)

fig7 = go.Figure(go.Bar(
    x=canal_df["Canal"], y=canal_df["pct"],
    marker=dict(color=[VERDE_A, CORAL, MOSTAZA], line=dict(color="white", width=2)),
    text=[f"{v}%" for v in canal_df["pct"]],
    textposition="outside",
    textfont=dict(size=14, color=GRIS_T),
    width=0.45,
))

fig7.update_layout(**base_layout("Mix de ventas promedio por canal", height=400, showlegend=False),
                   xaxis=dict(showgrid=False),
                   yaxis=dict(title="% promedio", showgrid=True,
                              gridcolor="#E8E8E8", range=[0, 100], zeroline=False))
fig7.show()
fig7.write_html(os.path.join(OUTPUT_PATH, "G7_mix_ventas.html"))
print("✓ G7 guardado")


✓ G7 guardado


## G8 · Brecha acceso vs. uso de datos de delivery
*Escena 6 — dos barras comparativas con flecha de brecha*

In [15]:

# ── G8 · Brecha acceso vs. uso de datos delivery (E6) ────────────────────────
# p12b: 0=No/NS, 1=Disponible, 2=Consulta activa
acceso = (df["p12b - Uso datos delivery (0=No/NS, 1=Disponible, 2=Consulta activa)"] >= 1).sum()
uso    = (df["p12b - Uso datos delivery (0=No/NS, 1=Disponible, 2=Consulta activa)"] == 2).sum()
n = len(df)

pct_acceso = round(acceso/n*100, 1)
pct_uso    = round(uso/n*100, 1)

fig8 = go.Figure()
fig8.add_trace(go.Bar(
    x=["Accede a datos", "Los usa activamente"],
    y=[pct_acceso, pct_uso],
    marker=dict(color=[AZUL_P, VERDE_A], line=dict(color="white", width=2)),
    text=[f"{pct_acceso}%", f"{pct_uso}%"],
    textposition="outside",
    textfont=dict(size=18, color=GRIS_T, family="Inter, Arial, sans-serif"),
    width=0.35,
))

# Flecha de brecha
fig8.add_annotation(
    ax=0, ay=pct_acceso/2 + 5, axref="x", ayref="y",
    x=1, y=pct_uso/2 + 5, xref="x", yref="y",
    showarrow=True, arrowhead=2, arrowsize=1.5, arrowwidth=2,
    arrowcolor=CORAL,
    text=f"  Brecha: {round(pct_acceso - pct_uso, 1)} pp",
    font=dict(color=CORAL, size=13),
)

fig8.update_layout(**base_layout("Datos de delivery: ¿se accede o se usan?", height=400, showlegend=False),
                   xaxis=dict(showgrid=False),
                   yaxis=dict(showgrid=True, gridcolor="#E8E8E8",
                              range=[0, max(pct_acceso, 100)*1.2], zeroline=False,
                              title="% de restaurantes"))
fig8.show()
fig8.write_html(os.path.join(OUTPUT_PATH, "G8_brecha_delivery.html"))
print("✓ G8 guardado")


✓ G8 guardado


## G10 · Distribución IPO — Índice de Plataformas de Opinión
*Escena 7 — dona bajo/medio/alto*

In [16]:

# ── G10 · Distribución IPO (E7) ──────────────────────────────────────────────
ipo_counts = df["Nivel IPO"].value_counts()
order_ipo = ["Alto (12-18)", "Medio (6-11)", "Bajo (0-5)"]
order_ipo_real = [o for o in order_ipo if o in ipo_counts.index]
ipo_df = pd.DataFrame({"Nivel": order_ipo_real,
                        "n": [ipo_counts.get(o, 0) for o in order_ipo_real]})
ipo_df["pct"] = (ipo_df["n"] / ipo_df["n"].sum() * 100).round(1)

fig10 = go.Figure(go.Pie(
    labels=ipo_df["Nivel"], values=ipo_df["n"],
    hole=0.60,
    marker=dict(colors=[VERDE_A, MOSTAZA, CORAL][:len(ipo_df)],
                line=dict(color="white", width=2.5)),
    textinfo="label+percent",
    textfont=dict(size=12, family="Inter, Arial, sans-serif"),
    sort=False,
))
fig10.add_annotation(text="<b>IPO</b><br>Plataformas<br>de opinión",
                     x=0.5, y=0.5, showarrow=False,
                     font=dict(size=13, color=GRIS_T, family="Inter, Arial, sans-serif"),
                     align="center")
fig10.update_layout(**base_layout("Índice de Plataformas de Opinión — distribución",
                                   height=420, showlegend=True))
fig10.show()
fig10.write_html(os.path.join(OUTPUT_PATH, "G10_dona_IPO.html"))
print("✓ G10 guardado")


✓ G10 guardado


## G13 · Adopción de IA y herramientas más usadas
*Escena 8 — indicador + barras*

In [17]:
# ── G13 · % adopción IA activa (E8) ──────────────────────────────────────────

# Conversión local por si el setup no la aplicó
col_ia = "Adopta alguna herramienta de IA (dummy)"
if df[col_ia].dtype == object:
    df[col_ia] = (df[col_ia] == "Si").astype(int)

for col in ["ia_chatgpt", "ia_claude", "ia_gemini", "ia_deepseek", "ia_copilot", "ia_integrad"]:
    if df[col].dtype == object:
        df[col] = (df[col] == "Si").astype(int)

pct_ia = round(df[col_ia].mean() * 100, 1)

herramientas = {
    "ChatGPT":     df["ia_chatgpt"].sum(),
    "Claude":      df["ia_claude"].sum(),
    "Gemini":      df["ia_gemini"].sum(),
    "DeepSeek":    df["ia_deepseek"].sum(),
    "Copilot":     df["ia_copilot"].sum(),
    "IA integrada (Canva AI, Meta AI...)": df["ia_integrad"].sum(),
}

h_df = pd.DataFrame(list(herramientas.items()), columns=["Herramienta", "n"])
h_df["pct"] = (h_df["n"] / len(df) * 100).round(1)
h_df = h_df[h_df["n"] > 0].sort_values("pct", ascending=True)

# ── Gráfico A: dona indicador ─────────────────────────────────────────────────
fig13a = go.Figure(go.Pie(
    values=[pct_ia, 100 - pct_ia],
    labels=["Usa IA", "No usa"],
    hole=0.68,
    marker=dict(colors=[VERDE_A, "#E8E8E8"], line=dict(color="white", width=3)),
    textinfo="none", sort=False,
))
fig13a.add_annotation(
    text=f"<b>{pct_ia}%</b><br><span style='font-size:13px'>usa IA</span>",
    x=0.5, y=0.5, showarrow=False,
    font=dict(size=30, color=GRIS_T, family="Inter, Arial, sans-serif"),
    align="center"
)
fig13a.update_layout(
    **base_layout("Adopción de herramientas de IA", height=380, showlegend=False),
)
fig13a.update_layout(margin=dict(l=80, r=80, t=72, b=48))

fig13a.show()
fig13a.write_html(os.path.join(OUTPUT_PATH, "G13a_adopcion_IA_dona.html"))
print("✓ G13a guardado")

# ── Gráfico B: barras herramientas ────────────────────────────────────────────
fig13b = go.Figure(go.Bar(
    x=h_df["pct"], y=h_df["Herramienta"],
    orientation="h",
    marker=dict(color=AZUL_P, line=dict(color="white", width=1)),
    text=[f"{v}%" for v in h_df["pct"]],
    textposition="outside",
    textfont=dict(size=13, color=GRIS_T),
    width=0.55,
))
fig13b.update_layout(
    **base_layout("Herramientas de IA más usadas", height=380, showlegend=False),
    xaxis=dict(showgrid=True, gridcolor="#E8E8E8",
               range=[0, h_df["pct"].max() * 1.3], zeroline=False,
               title="% de restaurantes"),
    yaxis=dict(showgrid=False),
)
fig13b.show()
fig13b.write_html(os.path.join(OUTPUT_PATH, "G13b_herramientas_IA.html"))
print("✓ G13b guardado")

✓ G13a guardado


✓ G13b guardado


## G14 · Acciones digitales planificadas (P14)
*Escena 8 — barras horizontales*

In [18]:

# ── G14 · Acciones digitales futuras P14 (E8) ────────────────────────────────
# P14 es texto libre con múltiples opciones separadas por coma
all_acciones = df["p14 - Acciones digitales a implementar en los proximos 12 meses"].dropna()
from collections import Counter
tokens = []
for row in all_acciones:
    tokens.extend([a.strip() for a in str(row).split(",")])

# Simplificar etiquetas largas
def shorten(s):
    s = s.strip()
    if "publicidad paga" in s.lower() or "ads" in s.lower(): return "Más publicidad paga (ads)"
    if "redes sociales" in s.lower() and "presencia" in s.lower(): return "Fortalecer presencia en redes"
    if "delivery" in s.lower(): return "Ampliar presencia en delivery"
    if "reseñas" in s.lower() or "google maps" in s.lower(): return "Gestión de reseñas"
    if "inteligencia artificial" in s.lower() or "ia" in s.lower(): return "Adoptar herramientas de IA"
    if "sitio web" in s.lower() or "seo" in s.lower(): return "Sitio web / SEO"
    if "fotografía" in s.lower() or "foto" in s.lower(): return "Fotografía profesional"
    return s[:40]

tokens_clean = [shorten(t) for t in tokens if len(t) > 3]
cnt = Counter(tokens_clean)
acc_df = pd.DataFrame(cnt.most_common(8), columns=["Acción", "n"])
acc_df["pct"] = (acc_df["n"] / len(df) * 100).round(1)
acc_df = acc_df.sort_values("pct", ascending=True)

colors_14 = [AZUL_C if i < len(acc_df)-1 else VERDE_A for i in range(len(acc_df))]

fig14 = go.Figure(go.Bar(
    x=acc_df["pct"], y=acc_df["Acción"],
    orientation="h",
    marker=dict(color=colors_14, line=dict(color="white", width=1)),
    text=[f"{v}%" for v in acc_df["pct"]],
    textposition="outside",
    textfont=dict(size=12, color=GRIS_T),
))
fig14.update_layout(**base_layout("Acciones digitales planificadas en los próximos 12 meses",
                                   height=460, showlegend=False),
                    xaxis=dict(showgrid=True, gridcolor="#E8E8E8",
                               range=[0, acc_df["pct"].max()*1.3], zeroline=False,
                               title="% de restaurantes"),
                    yaxis=dict(showgrid=False))
fig14.show()
fig14.write_html(os.path.join(OUTPUT_PATH, "G14_acciones_futuras.html"))
print("✓ G14 guardado")


✓ G14 guardado


## G15 · Factor más determinante para posicionamiento futuro (P16)
*Escena 8 — barras horizontales*

In [19]:

# ── G15 · Factor más determinante para posicionamiento futuro P16 (E8) ────────
p16 = df["p16 - Factor mas determinante para posicionamiento futuro en nicho mediterraneo"].dropna()

def shorten16(s):
    s = str(s).strip()
    if "calidad" in s.lower() or "autenticidad" in s.lower(): return "Calidad y autenticidad"
    if "reputación" in s.lower() or "presencia" in s.lower(): return "Presencia y reputación digital"
    if "experiencia" in s.lower() or "ambiente" in s.lower(): return "Experiencia y ambiente"
    if "precio" in s.lower(): return "Precio competitivo"
    if "ubicación" in s.lower(): return "Ubicación"
    if "inteligencia" in s.lower() or "ia" in s.lower(): return "Adopción de IA"
    return s[:35]

p16_clean = p16.apply(shorten16)
p16_df = p16_clean.value_counts().reset_index()
p16_df.columns = ["Factor", "n"]
p16_df["pct"] = (p16_df["n"] / len(df) * 100).round(1)
p16_df = p16_df.sort_values("pct", ascending=True)

colors_16 = [CORAL if v == p16_df["pct"].max() else AZUL_C for v in p16_df["pct"]]

fig15 = go.Figure(go.Bar(
    x=p16_df["pct"], y=p16_df["Factor"],
    orientation="h",
    marker=dict(color=colors_16, line=dict(color="white", width=1)),
    text=[f"{v}%" for v in p16_df["pct"]],
    textposition="outside",
    textfont=dict(size=12, color=GRIS_T),
))
fig15.update_layout(**base_layout("Factor más determinante para el posicionamiento futuro",
                                   height=420, showlegend=False),
                    xaxis=dict(showgrid=True, gridcolor="#E8E8E8",
                               range=[0, p16_df["pct"].max()*1.3], zeroline=False,
                               title="% de restaurantes"),
                    yaxis=dict(showgrid=False))
fig15.show()
fig15.write_html(os.path.join(OUTPUT_PATH, "G15_factor_futuro.html"))
print("✓ G15 guardado")


✓ G15 guardado


## G16 · Aplicación de IA de mayor interés (P18)
*Escena 8 — barras horizontales*

In [20]:

# ── G16 · App de IA de mayor interés P18 (E8) ────────────────────────────────
p18 = df["p18 - Aplicacion de IA de mayor interes para el restaurante"].dropna()

def shorten18(s):
    s = str(s).strip()
    if "pauta" in s.lower() or "publicitaria" in s.lower(): return "Pauta publicitaria"
    if "menú" in s.lower() or "recomendaciones" in s.lower(): return "Recomendaciones de menú"
    if "atención" in s.lower() or "chatbot" in s.lower(): return "Atención al cliente / chatbot"
    if "reseñas" in s.lower(): return "Gestión de reseñas"
    if "análisis" in s.lower() or "analitica" in s.lower(): return "Análisis de datos"
    if "contenido" in s.lower(): return "Generación de contenido"
    return s[:35]

p18_clean = p18.apply(shorten18)
p18_df = p18_clean.value_counts().reset_index()
p18_df.columns = ["App", "n"]
p18_df["pct"] = (p18_df["n"] / len(df) * 100).round(1)
p18_df = p18_df.sort_values("pct", ascending=True)

colors_18 = [VERDE_A if v == p18_df["pct"].max() else AZUL_C for v in p18_df["pct"]]

fig16 = go.Figure(go.Bar(
    x=p18_df["pct"], y=p18_df["App"],
    orientation="h",
    marker=dict(color=colors_18, line=dict(color="white", width=1)),
    text=[f"{v}%" for v in p18_df["pct"]],
    textposition="outside",
    textfont=dict(size=12, color=GRIS_T),
))
fig16.update_layout(**base_layout("Aplicación de IA de mayor interés", height=400, showlegend=False),
                    xaxis=dict(showgrid=True, gridcolor="#E8E8E8",
                               range=[0, p18_df["pct"].max()*1.3], zeroline=False,
                               title="% de restaurantes"),
                    yaxis=dict(showgrid=False))
fig16.show()
fig16.write_html(os.path.join(OUTPUT_PATH, "G16_app_ia.html"))
print("✓ G16 guardado")


✓ G16 guardado


## G17 · Brecha intención vs. adopción de IA
*Escena 8 — dos barras con línea de brecha*

In [21]:
# ── G17 · Brecha intención vs. adopción IA (E8) ──────────────────────────────
intencion = df["p14 - Acciones digitales a implementar en los proximos 12 meses"].dropna().apply(
    lambda x: 1 if "inteligencia artificial" in str(x).lower() else 0
).sum()

adopcion = (df["Adopta alguna herramienta de IA (dummy)"] == "Si").sum()
n = len(df)

pct_int = round(intencion/n*100, 1)
pct_ado = round(adopcion/n*100, 1)

fig17 = go.Figure()
fig17.add_trace(go.Bar(
    x=["Intención de adoptar IA<br>(próximos 12 meses)", "Uso efectivo actual de IA"],
    y=[pct_int, pct_ado],
    marker=dict(color=[AZUL_P, VERDE_A], line=dict(color="white", width=2)),
    text=[f"{pct_int}%", f"{pct_ado}%"],
    textposition="outside",
    textfont=dict(size=18, color=GRIS_T, family="Inter, Arial, sans-serif"),
    width=0.38,
))

# Brecha
mid = (pct_int + pct_ado) / 2
fig17.add_shape(type="line",
    x0=0, y0=pct_int, x1=1, y1=pct_ado,
    line=dict(color=CORAL, width=2, dash="dot"))
fig17.add_annotation(
    x=0.5, y=mid + 4, text=f"Brecha: {round(pct_int - pct_ado, 1)} pp",
    showarrow=False, font=dict(color=CORAL, size=13, family="Inter, Arial, sans-serif"),
    align="center")

fig17.update_layout(**base_layout("Brecha intención vs. adopción de IA", height=420, showlegend=False),
                    xaxis=dict(showgrid=False),
                    yaxis=dict(showgrid=True, gridcolor="#E8E8E8",
                               range=[0, max(pct_int, pct_ado)*1.35],
                               zeroline=False, title="% de restaurantes"))
fig17.show()
fig17.write_html(os.path.join(OUTPUT_PATH, "G17_brecha_IA.html"))
print("✓ G17 guardado")


✓ G17 guardado


## 📋 Tabla 3 · Calcular todos los [DATO] del guion

In [23]:
# ── TABLA 3 · Calcular todos los [DATO] del guion ────────────────────────────
print("=" * 60)
print("ESTADÍSTICAS PARA REEMPLAZAR [DATO] EN EL GUION")
print("=" * 60)

n = len(df)

# E1
e1 = (df["p4 - Realiza inversion en marketing digital"] == "Si").sum()
print(f"\n[E5-1] % invierte en mkt digital:       {round(e1/n*100,1)}%")

# E5-2: top 3 plataformas
p = {
    "FB/IG Ads": (df["p5 - FB/IG Ads (dummy)"] == "Si").sum(),
    "Google Ads": (df["p5 - Google Ads (dummy)"] == "Si").sum(),
    "TikTok Ads": (df["p5 - TikTok Ads (dummy)"] == "Si").sum()
}
top3 = sorted(p.items(), key=lambda x: x[1], reverse=True)[:3]
print(f"[E5-2] Top 3 plataformas:                {', '.join([f'{k} ({round(v/n*100,0):.0f}%)' for k,v in top3])}")

# E5-3: distribución IRS
for lvl in df["Nivel IRS"].value_counts(normalize=True).items():
    print(f"[E5-3] IRS {lvl[0]}:                 {round(lvl[1]*100,1)}%")

# E6-4: delivery de cada 10
print(f"\n[E6-4] Clientes delivery (de cada 10):  {round(df['p10 - Clientes delivery normalizados (de cada 10)'].mean(),1)}")

# E6-5/6: acceso y uso datos delivery
acc = (df["p12b - Uso datos delivery (0=No/NS, 1=Disponible, 2=Consulta activa)"] >= 1).sum()
uso = (df["p12b - Uso datos delivery (0=No/NS, 1=Disponible, 2=Consulta activa)"] == 2).sum()
print(f"[E6-5] Accede a datos delivery:          {round(acc/n*100,1)}%")
print(f"[E6-6] Usa datos activamente:            {round(uso/n*100,1)}%")

# E7
print(f"\n[E7-8] Promedio reseñas Google Maps:    {round(df['obs1 - Numero de resenas en Google Maps'].mean(),0):.0f}")
print(f"[E7-9] Calificación promedio Google:     {round(df['obs2 - Calificacion promedio en Google Maps (1.0-5.0)'].mean(),2)}")
ta = df["obs3 - Presencia activa en TripAdvisor (1=Si, 0=No)"].sum()
print(f"[E7-10] % con perfil TripAdvisor:        {round(ta/n*100,1)}%")

for lvl in df["Nivel IPO"].value_counts(normalize=True).items():
    print(f"[E7-11] IPO {lvl[0]}:                {round(lvl[1]*100,1)}%")

# E8
df["Adopta alguna herramienta de IA (dummy)"] = (
    df["Adopta alguna herramienta de IA (dummy)"] == "Si"
).astype(int)

n = len(df)
ia = df["Adopta alguna herramienta de IA (dummy)"].sum()
print(f"\n[E8-12] % usa alguna IA:                {round(ia/n*100,1)}%")
p18_moda = df["p18 - Aplicacion de IA de mayor interes para el restaurante"].mode()[0]
print(f"[E8-13] App IA de mayor interés:         {p18_moda}")
p16_rep = df["p16 - Factor mas determinante para posicionamiento futuro en nicho mediterraneo"].value_counts(normalize=True).iloc[0]
p16_lbl = df["p16 - Factor mas determinante para posicionamiento futuro en nicho mediterraneo"].value_counts().index[0]
print(f"[E8-14] Factor futuro más frecuente:     {p16_lbl} ({round(p16_rep*100,1)}%)")

# E8-17: brecha intención vs adopción
int_ia = df["p14 - Acciones digitales a implementar en los proximos 12 meses"].dropna().apply(
    lambda x: 1 if "inteligencia artificial" in str(x).lower() else 0).sum()
print(f"\n[E8-17] Intención adoptar IA (P14):     {round(int_ia/n*100,1)}%")
print(f"[E8-17] Uso efectivo IA (actual):        {round(ia/n*100,1)}%")

# Mix ventas
print(f"\n[E6-16] Mix ventas promedio:")
print(f"  Presencial:      {round(df['p9 - Clientes presenciales normalizados (de cada 10)'].mean()*10,1)}%")
print(f"  Delivery:        {round(df['p10 - Clientes delivery normalizados (de cada 10)'].mean()*10,1)}%")
print(f"  Domicilio propio:{round(df['p11 - Clientes domicilio propio normalizados (de cada 10)'].mean()*10,1)}%")

# ── IDL: promedio general y por hot_zone ──────────────────────────────────────
# ⚠️ Ajusta estos nombres si difieren en tu base
COL_IDL      = "Indice de Delivery (0-10)"          # columna con el índice IDL
COL_HOT_ZONE = "hot_zone"     # columna con la clasificación de zona

print("\n" + "=" * 60)
print("IDL — ÍNDICE DE DIGITALIZACIÓN LOCAL")
print("=" * 60)

# — Promedio general
idl_general = df[COL_IDL].mean()
idl_median  = df[COL_IDL].median()
print(f"\n[IDL-G] Promedio general IDL:            {round(idl_general, 2)}")
print(f"[IDL-G] Mediana  general IDL:            {round(idl_median,  2)}")

# — Promedio por hot_zone
idl_zona = (
    df.groupby(COL_HOT_ZONE)[COL_IDL]
      .agg(n="count", media="mean", mediana="median", std="std")
      .round(2)
      .sort_values("media", ascending=False)
)

print(f"\n[IDL-Z] IDL promedio por zona:")
print(f"  {'Zona':<20} {'n':>4}  {'Media':>6}  {'Mediana':>7}  {'Std':>5}")
print(f"  {'-'*20} {'-'*4}  {'-'*6}  {'-'*7}  {'-'*5}")
for zona, row in idl_zona.iterrows():
    print(f"  {str(zona):<20} {int(row['n']):>4}  {row['media']:>6.2f}  {row['mediana']:>7.2f}  {row['std']:>5.2f}")

# — Brecha concentración vs dispersión
#   Asume que hot_zone tiene al menos dos categorías; ajusta los labels si difieren
try:
    zonas_unicas = idl_zona.index.tolist()
    zona_alta    = zonas_unicas[0]   # mayor IDL promedio (concentración)
    zona_baja    = zonas_unicas[-1]  # menor IDL promedio (dispersión)
    brecha       = idl_zona.loc[zona_alta, "media"] - idl_zona.loc[zona_baja, "media"]
    print(f"\n[IDL-B] Brecha concentración vs dispersión:")
    print(f"  Zona alta  → {zona_alta:<20} IDL = {idl_zona.loc[zona_alta, 'media']:.2f}")
    print(f"  Zona baja  → {zona_baja:<20} IDL = {idl_zona.loc[zona_baja,  'media']:.2f}")
    print(f"  Diferencia → {round(brecha, 2)} puntos")
except Exception as e:
    print(f"  ⚠️  No se pudo calcular brecha: {e}")

print("=" * 60)

ESTADÍSTICAS PARA REEMPLAZAR [DATO] EN EL GUION

[E5-1] % invierte en mkt digital:       80.0%
[E5-2] Top 3 plataformas:                FB/IG Ads (69%), Google Ads (23%), TikTok Ads (17%)
[E5-3] IRS Medio (5-9):                 48.6%
[E5-3] IRS Alto (10-14):                 25.7%
[E5-3] IRS Bajo (0-4):                 25.7%

[E6-4] Clientes delivery (de cada 10):  1.4
[E6-5] Accede a datos delivery:          34.3%
[E6-6] Usa datos activamente:            14.3%

[E7-8] Promedio reseñas Google Maps:    1316
[E7-9] Calificación promedio Google:     4.52
[E7-10] % con perfil TripAdvisor:        82.9%
[E7-11] IPO Medio (6-11):                48.6%
[E7-11] IPO Alto (12-18):                40.0%
[E7-11] IPO Bajo (0-5):                11.4%

[E8-12] % usa alguna IA:                0.0%
[E8-13] App IA de mayor interés:         Crear pauta publicitaria optimizada
[E8-14] Factor futuro más frecuente:     Calidad y autenticidad del producto (40.0%)

[E8-17] Intención adoptar IA (P14):     45.7%
[E